In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [4]:
import json

# ---- 工具定义 ----
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"}
                },
                "required": ["city"]
            }
        }
    }
]

# ---- 工具实现 ----
def get_weather(city: str) -> str:
    fake_data = {"北京": "晴, 28°C", "上海": "多云, 32°C"}
    return fake_data.get(city, f"{city}：数据暂无")

# ---- 工具注册 ----
tool_map = {"get_weather": get_weather}

# ---- Agent 核心循环 ----
def run_agent(user_input: str, max_iterations: int = 10):
    messages = [
        {"role": "system", "content": "你是一个有用的助手，可以查询天气。"},
        {"role": "user", "content": user_input}
    ]

    for i in range(max_iterations):
        response = client.chat.completions.create(
            model="qwen3.5-flash",
            messages=messages,
            tools=tools,
        )
        choice = response.choices[0]
        msg = choice.message

        # 打印思考过程
        if msg.reasoning_content:
            print(f"[思考] {msg.reasoning_content}")

        # 打印工具调用详情
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[工具调用] id={tc.id}, name={tc.function.name}, arguments={tc.function.arguments}")

        # 必须先把 assistant 消息追加到历史，再追加工具结果
        messages.append(msg)

        # 检查是否有工具调用
        if not msg.tool_calls:
            # 没有工具调用 → 循环结束
            print(f"[Agent] 循环 {i+1} 轮后完成")
            return msg.content

        # 执行工具调用
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"[Agent] 执行: {fn_name}({fn_args})")

            result = tool_map[fn_name](**fn_args)
            print(f"[Agent] 结果: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    return "达到最大迭代次数，Agent 停止"

# ---- 运行 ----
answer = run_agent("北京今天天气怎么样？如果超过30度推荐冷饮")
print(f"\n最终回答：{answer}")

[思考] 用户想查询北京今天的天气，并且说如果温度超过30度就推荐冷饮。我需要先调用get_weather工具来查询北京的天气情况。

让我查看一下工具参数：
- city: 城市名称（必需参数）

用户提供了城市"北京"，所以我可以调用这个工具。
[工具调用] id=call_f5bb896e78a14e06b5161e2e, name=get_weather, arguments={"city": "北京"}
[Agent] 执行: get_weather({'city': '北京'})
[Agent] 结果: 晴, 28°C
[思考] 北京今天的天气是晴天，温度28°C。用户说如果超过30度就推荐冷饮，但现在是28°C，没有超过30度，所以我不需要推荐冷饮。我只需要告诉用户北京的天气情况即可。
[Agent] 循环 2 轮后完成

最终回答：北京今天的天气是晴天，气温为28°C。由于温度未达到30度以上，暂时不需要特别推荐冷饮哦！今天是个适合户外活动的好天气～
